# 10 · CP4 fix: the CP1 shortcut checks on the new sampler, seed 42 (CPU)

**CPU runtime** (no GPU; about 0.08 compute units per hour). Run it **before** the retrain (notebook 11).

**The fix (Dhanush, 17 Nov 2026):** for edited images with an AI-generated base, 1/3 of the training draws take a crop anywhere in the image.
- A crop with **no edited pixel** is labelled **generated**, with an empty mask. The loader re-checks this against the edit mask.
- A crop with **1–90%** edited pixels stays **edited**.
- Anything in between is redrawn.

So some DiffSeg30k files move from the edited class to the generated class.

**What this notebook checks, on the trainer's own draws** (epochs 0–2, 72,000 draws per sampler), comparing the CP3 sampler with the fixed one:
- the CP1 metadata classifier: a lookup table on format, JPEG quality and crop size, as CP1 defined it, scored as 5-fold balanced accuracy. Real vs not real must stay ≤ 0.60; the 3-class score is reported. The stored image's own size is shown beside it as a diagnostic, as CP1 showed the original size;
- the quality-75 share per class, and per class and source;
- the source mix per class. DiffSeg30k entering the generated class is the designed change, so it is reported on its own, and materiality is judged on the mix without the edit-free context crops;
- CP1 run 5's recorded numbers, shown for reference (they come from CP1's own crops).

**It stops** if anything moves materially:
- real vs not real above 0.60;
- a balanced accuracy changing by more than 0.02;
- a quality-75 share or a class/source share changing by more than 0.03.

Then Dhanush decides. No model runs here.

**Result** (send to Claude): `Paper 1/runs/checkpoints/cp4_fix_sampler_check.json`.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings
import datetime, json, logging, os, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
from eviground.config import load_config
from eviground.data import pack
from eviground.env import get_paths, session_info

SEED = 42
CONTEXT_SHARE = 0.3333     # Dhanush, 17 Nov 2026: 1/3 of the edited AI-base draws
EPOCHS = range(3)          # the first three epoch plans of each sampler
CU_PER_HOUR = 0.08         # CPU runtime: read the live rate in Colab's resource panel
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:   # fake world, fake shards
    import shutil, tempfile
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_cp4fix_check_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    EPOCHS = range(2)
paths.ensure()
CONFIG = 'debug' if DRY else 'base'
cfg = load_config(CONFIG)
PROFILE = 'unified'
N_PER_CLASS = int(cfg.train.crops_per_epoch) // 3 if not DRY else 200
CK = paths.runs / 'checkpoints'
print('commit', info['git_commit'], '| draws per epoch', 3 * N_PER_CLASS, '| epochs', list(EPOCHS),
      '| context share', CONTEXT_SHARE)

In [ ]:
# Cell 3: the shards (dry run: fake world -> fake CP1 run -> pack), copied to the local disk
SHARDS = paths.shards
if DRY:
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    out = paths.runs / 'checkpoints' / 'cp1'
    run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20,
                             base_unclear_max=0.2, diffseg_card_splits={'train': 18, 'validation': 12}),
            out_dir=out, cache_dir=paths.cache / 'cp1', progress=None)
    pack.pack(out / 'manifest_cp1.csv.gz', SHARDS, free_bytes=10**12, margin=0,
              shard_bytes=200_000, sample_per_source=5, progress=None)
done = pack.read_done(SHARDS)
assert done, f'no finished shards in {SHARDS}: run notebook 04 first'
print('shards:', done['images'], 'images |', done['mask_rule'])
# each read through Colab's Drive mount costs ~0.25 s: train from a local copy (checked shard by shard)
LOCAL_SHARDS = paths.local_data / 'shards'          # /content/data/shards on Colab
staged = pack.stage_shards(SHARDS, LOCAL_SHARDS, progress=print)
SHARDS = LOCAL_SHARDS
INDEX_SHA256 = pack.file_sha256(SHARDS / 'index.parquet')
print('staged:', staged)

In [ ]:
# Cell 4: both samplers, the shortcut checks and their comparison
from eviground.data import sampler_check
from eviground.data.datasets import ForensicDataset

t = time.time()
RESULTS = {}
for name, share in (('cp3', 0.0), ('fix', CONTEXT_SHARE)):
    ds = ForensicDataset(SHARDS, 'train', profile=PROFILE, seed=SEED, n_per_class=N_PER_CLASS,
                         context_share=share)
    records = sampler_check.plan_records(ds, EPOCHS)
    RESULTS[name] = sampler_check.check(records, seed=SEED)
    print(name, '| draws', RESULTS[name]['draws'], '| classes', RESULTS[name]['class_counts'],
          '| context', RESULTS[name]['context'], '| real vs not real',
          RESULTS[name]['criterion_score'], '| 3-class', RESULTS[name]['metadata_three_class']['balanced_accuracy'],
          '| stored size (diagnostic)', RESULTS[name]['diagnostic_stored_size_real_vs_not_real'])
COMPARE = sampler_check.compare(RESULTS['cp3'], RESULTS['fix'])
for c in ('real', 'generated', 'edited'):
    print(f"{c:9} q75 cp3 {RESULTS['cp3']['quality_75']['per_class'][c]} | fix {RESULTS['fix']['quality_75']['per_class'][c]}")
    print(f"{'':9} sources cp3 {RESULTS['cp3']['source_mix'][c]}\n{'':9} sources fix {RESULTS['fix']['source_mix'][c]}")
REFERENCE = None
ref_file = CK / 'cp1_data_integrity.json'
if ref_file.is_file():   # CP1 run 5, for reference (its own crops, not the trainer's draws)
    res = json.loads(ref_file.read_text()).get('result', {})
    meta, q75 = res.get('metadata_classifier') or {}, res.get('quality_75') or {}
    REFERENCE = {'criterion_score': meta.get('criterion_score'),
                 'three_class': (meta.get('criterion') or {}).get('three_class', {}).get('balanced_accuracy'),
                 'quality_75_per_class': q75.get('per_class')}
print('CP1 run 5 (reference):', REFERENCE)
print('material changes:', COMPARE['material'] or 'none')
print(f'{time.time() - t:.0f} s')

In [ ]:
# Cell 5: the result file; stop if anything moved materially
hours = (time.time() - T0) / 3600
RESULT = {
    'step': 'CP4 fix: CP1 shortcut checks on the training sampler', 'dry_run': DRY, 'branch': BRANCH,
    'seed': SEED, 'session': info, 'context_share': CONTEXT_SHARE, 'epochs': list(EPOCHS),
    'n_per_class': N_PER_CLASS, 'index_sha256': pack.file_sha256(SHARDS / 'index.parquet'),
    'cp3_sampler': RESULTS['cp3'], 'fix_sampler': RESULTS['fix'], 'compare': COMPARE,
    'cp1_run5_reference': REFERENCE,
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'wall_time_h': round(hours, 3), 'cu_per_hour': CU_PER_HOUR, 'compute_units': round(hours * CU_PER_HOUR, 3),
}
out = CK / 'cp4_fix_sampler_check.json'
out.write_text(json.dumps(RESULT, indent=1, default=str))
print('saved', out, '| CU', RESULT['compute_units'])
if DRY:
    assert RESULTS['cp3']['context'] == {} and RESULTS['fix']['draws'] == RESULTS['cp3']['draws']
elif COMPARE['material']:
    raise RuntimeError('the fix moves a shortcut check materially: stop and ask Dhanush. '
                       f"{COMPARE['material']} (send {out} to Claude)")